In [1]:
import pprint
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [2]:
model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=100, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_ollama = init_chat_model("ollama:gemma4:latest",
                                max_tokens=200, 
                                temperature=0.0)

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

from langchain.agents import create_agent

from langchain.agents.structured_output import ToolStrategy

In [4]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class ProductReview:
    """Analysis of a product review."""
    product_name: str = field(metadata={"description": "The name of the product"})
    review_text: str = field(metadata={"description": "The text of the review"})
    rating: Literal[1, 2, 3, 4, 5] = field(metadata={"description": "The rating given by the reviewer"})
    shipping_sentiment: Literal["positive", "negative", "neutral"] = field(metadata={"description": "The sentiment of the review regarding shipping"})
    pricing_sentiment: Literal["positive", "negative", "neutral"] = field(metadata={"description": "The sentiment of the review regarding pricing"})
    quality_sentiment: Literal["positive", "negative", "neutral"] = field(metadata={"description": "The sentiment of the review regarding quality"})

In [5]:
agent = create_agent(
    model=model_gr_lamma,
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

In [6]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great Monitor: 3 out of 5 stars. slow shipping, but less expensive'"}]
})

In [7]:
from pprint import pprint

pprint(result, depth=4, compact=True)

{'messages': [HumanMessage(content="Analyze this review: 'Great Monitor: 3 out of 5 stars. slow shipping, but less expensive'", additional_kwargs={}, response_metadata={}, id='a9f93f33-c4c8-4fd5-b476-04116fe07795'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'mgm35tm53', 'function': {'arguments': '{"pricing_sentiment":"positive","product_name":"Monitor","quality_sentiment":"neutral","rating":3,"review_text":"Great Monitor: 3 out of 5 stars. slow shipping, but less expensive","shipping_sentiment":"negative"}', 'name': 'ProductReview'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 397, 'total_tokens': 463, 'completion_time': 0.125245652, 'completion_tokens_details': None, 'prompt_time': 0.042367569, 'prompt_tokens_details': None, 'queue_time': 0.16170966, 'total_time': 0.167613221}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_re

In [8]:
result["structured_response"]

ProductReview(product_name='Monitor', review_text='Great Monitor: 3 out of 5 stars. slow shipping, but less expensive', rating=3, shipping_sentiment='negative', pricing_sentiment='positive', quality_sentiment='neutral')

In [13]:
from dataclasses import asdict
asdict(result["structured_response"])

{'product_name': 'Monitor',
 'review_text': 'Great Monitor: 3 out of 5 stars. slow shipping, but less expensive',
 'rating': 3,
 'shipping_sentiment': 'negative',
 'pricing_sentiment': 'positive',
 'quality_sentiment': 'neutral'}